<a href="https://colab.research.google.com/github/DiegoB2002/BUS4-118S/blob/dev/PromptengineeringExercise3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3

In [ ]:
# ============================================================
# Exercise 3 — Self-Reflection (Critique-and-Revise) Template
# NO API KEY required (rule-based "fake LLM" reviewer + reviser)
# Produces visible BEFORE/AFTER outputs + clear differences
# ============================================================

import re
import json
from typing import List, Dict, Any

# -----------------------------
# 1) FILL THESE IN (your assignment content)
# -----------------------------
TASK_OBJECTIVE = "Summarize the announcement for executives, focusing on business impact."

TARGET_AUDIENCE = "Executives (non-technical)"
LENGTH_LIMIT_WORDS = 120
FORMAT_REQUIREMENT = "Single paragraph (no bullet points)."

REQUIREMENTS: List[str] = [
    "Focus on business impact (not implementation details)",
    "Mention timeline",
    "Mention risk level",
    "Mention revenue implications",
    "Avoid technical jargon and internal engineering terms",
]

BEFORE_SUMMARY = """The engineering team released a new backend optimization that improves database indexing and caching performance.
It reduces API latency by 40% and refactors the query engine for better efficiency.
Several internal components were updated, and the system architecture was modified to support scaling.
Testing was successful, and monitoring is in place.
The rollout will happen gradually over the next few weeks.
""".strip()


# -----------------------------
# Helpers
# -----------------------------
def word_count(s: str) -> int:
    return len(re.findall(r"\b\w+\b", s or ""))

def is_single_paragraph(s: str) -> bool:
    # Treat multiple blank-line separated blocks as multiple paragraphs
    blocks = [b for b in re.split(r"\n\s*\n", (s or "").strip()) if b.strip()]
    return len(blocks) <= 1

def looks_too_technical(s: str) -> bool:
    # Very simple heuristic. Add/adjust terms if needed.
    tech_terms = [
        "backend", "database", "index", "caching", "api", "latency", "refactor",
        "query engine", "architecture", "components", "scaling", "monitoring"
    ]
    t = s.lower()
    hits = [term for term in tech_terms if term in t]
    return len(hits) >= 3

def ensure_length(s: str, limit_words: int) -> str:
    words = re.findall(r"\S+", s.strip())
    if len(words) <= limit_words:
        return s.strip()
    return " ".join(words[:limit_words]).strip()

def ensure_paragraph_format(s: str, format_requirement: str) -> str:
    # If single-paragraph required, collapse line breaks into spaces.
    if "single paragraph" in format_requirement.lower():
        s = re.sub(r"\s*\n\s*", " ", s.strip())
        s = re.sub(r"\s+", " ", s).strip()
    return s

def jprint(x: Any):
    print(json.dumps(x, indent=2, ensure_ascii=False))


# -----------------------------
# 2) Self-reflection "prompt" (for your doc) — printed for transparency
# -----------------------------
SELF_REFLECTION_PROMPT = f"""
SYSTEM ROLE:
You are a Senior Editorial Reviewer performing structured self-reflection.

OBJECTIVE:
Critique and improve the summary so that it fully aligns with:
- Task objective
- Requirements checklist
- Target audience
- Length limit and formatting constraints

EVALUATION CRITERIA (explicit):
1) Accuracy (no invented facts; faithful to source/context)
2) Clarity (plain language; no ambiguity)
3) Audience Fit (appropriate tone/complexity for {TARGET_AUDIENCE})
4) Length Limits (<= {LENGTH_LIMIT_WORDS} words)
5) Formatting (must match: {FORMAT_REQUIREMENT})
6) Specificity (concrete and meaningful; avoids filler)
7) Goal Alignment (supports the task objective)
8) Completeness (covers all requirements; no critical omissions)

REQUIRED OUTPUTS (must include all):
1) BEFORE Summary (verbatim)
2) Criteria Assessment (PASS/PARTIAL/FAIL + explanation)
3) Major Gaps (3–5 bullets)
4) Revision Plan (bullets)
5) AFTER Summary (improved; must satisfy constraints)
6) Differences Explained (what changed + why)
""".strip()


# -----------------------------
# 3) Fake reviewer: critique against explicit criteria + requirements
# -----------------------------
def critique_summary(
    before: str,
    requirements: List[str],
    audience: str,
    length_limit: int,
    format_req: str,
    task_objective: str
) -> Dict[str, Any]:

    before_wc = word_count(before)
    single_para_ok = is_single_paragraph(before) if "single paragraph" in format_req.lower() else True
    technical = looks_too_technical(before)

    # Requirement coverage heuristic
    req_hits = {}
    text = before.lower()

    def hit_any(keywords):
        return any(k in text for k in keywords)

    req_hits["timeline"] = hit_any(["timeline", "next few weeks", "by", "rollout", "launch date", "gradually"])
    req_hits["risk"] = hit_any(["risk", "low risk", "medium risk", "high risk", "stable", "testing"])
    req_hits["revenue"] = hit_any(["revenue", "retention", "conversion", "sales", "growth", "upsell", "churn"])
    req_hits["business_impact"] = hit_any(["business", "customers", "user experience", "impact", "growth", "retention", "conversion"])
    req_hits["avoid_jargon"] = not technical

    # Criteria judgments (simple but explicit)
    criteria = []

    # Accuracy: with no source doc provided, we can only check "no new facts added" (can’t verify truth).
    criteria.append({
        "criterion": "Accuracy",
        "grade": "PASS",
        "explanation": "No obvious fabricated details beyond what the draft states; however, the source context is not provided so truth cannot be independently verified."
    })

    # Clarity
    criteria.append({
        "criterion": "Clarity",
        "grade": "PASS" if not technical else "PARTIAL",
        "explanation": "Readable, but leans on technical terms that may reduce clarity for non-technical readers." if technical else "Clear and easy to follow."
    })

    # Audience fit
    criteria.append({
        "criterion": "Audience Fit",
        "grade": "FAIL" if technical else "PASS",
        "explanation": f"Too technical for {audience}; focuses on implementation details." if technical else f"Appropriate for {audience}."
    })

    # Length limits
    criteria.append({
        "criterion": "Length Limits",
        "grade": "PASS" if before_wc <= length_limit else "FAIL",
        "explanation": f"{before_wc} words (limit {length_limit})."
    })

    # Formatting
    criteria.append({
        "criterion": "Formatting",
        "grade": "PASS" if single_para_ok else "FAIL",
        "explanation": "Matches required format." if single_para_ok else "Does not match single-paragraph requirement (multiple paragraphs detected)."
    })

    # Specificity (not just tech)
    criteria.append({
        "criterion": "Specificity",
        "grade": "PASS",
        "explanation": "Includes concrete detail (e.g., performance improvement and rollout), but could be more specific about business outcomes."
    })

    # Goal alignment
    goal_ok = req_hits["business_impact"] and req_hits["revenue"]
    criteria.append({
        "criterion": "Goal Alignment",
        "grade": "PASS" if goal_ok else "PARTIAL",
        "explanation": "Partially aligned; mentions performance but not the executive-level business framing and revenue implications." if not goal_ok else "Aligned with objective."
    })

    # Completeness (requirements)
    missing = []
    if not req_hits["business_impact"]:
        missing.append("business impact framing")
    if not req_hits["revenue"]:
        missing.append("revenue implications")
    if not req_hits["risk"]:
        missing.append("risk level")
    if not req_hits["timeline"]:
        missing.append("timeline")
    if technical:
        missing.append("avoid technical jargon")

    criteria.append({
        "criterion": "Completeness",
        "grade": "FAIL" if missing else "PASS",
        "explanation": f"Missing: {', '.join(missing)}." if missing else "Covers all required elements."
    })

    # Major gaps + revision plan
    major_gaps = []
    if technical:
        major_gaps.append("Uses engineering jargon/implementation details rather than business impact.")
    if not req_hits["revenue"]:
        major_gaps.append("Does not mention revenue implications (retention/conversion/growth).")
    if not req_hits["risk"]:
        major_gaps.append("No explicit risk level assessment.")
    if not req_hits["business_impact"]:
        major_gaps.append("Does not frame the update in executive-level outcomes.")
    if not single_para_ok:
        major_gaps.append("Formatting does not match the required single-paragraph format.")

    major_gaps = major_gaps[:5]

    revision_plan = [
        "Reframe the update in terms of customer experience and business impact (executive-friendly).",
        "Add revenue implication (e.g., retention/conversion/growth) without inventing numbers.",
        "State timeline clearly (e.g., phased rollout over the next few weeks).",
        "Add explicit risk level (low/medium/high) justified by what is stated (e.g., testing + phased rollout).",
        "Remove/replace technical jargon; keep one concrete metric only if helpful (e.g., 40% latency reduction).",
    ]

    # Keep plan short if fewer issues
    revision_plan = revision_plan[:5]

    return {
        "before_word_count": before_wc,
        "criteria_assessment": criteria,
        "major_gaps": major_gaps,
        "revision_plan": revision_plan
    }


# -----------------------------
# 4) Fake reviser: produce AFTER summary meeting constraints
# -----------------------------
def revise_summary(
    before: str,
    critique: Dict[str, Any],
    requirements: List[str],
    audience: str,
    length_limit: int,
    format_req: str,
    task_objective: str
) -> Dict[str, Any]:

    # Build an executive-friendly paragraph using ONLY info present in BEFORE,
    # plus safe business framing (no new facts, no new numbers).
    # Allowed: interpretive framing like "supports retention/conversion" (implication, not a factual claim).

    # Extract a concrete metric if present
    metric = ""
    m = re.search(r"(\d+%)[^\n]*latency", before.lower())
    if m:
        metric = m.group(1)

    # Timeline inference if mentioned
    timeline = "over the next few weeks"
    if "next few weeks" not in before.lower():
        # If not present, avoid asserting a new timeline.
        timeline = ""

    # Risk justification (only if testing + phased rollout mentioned)
    risk_level = ""
    if ("testing" in before.lower() or "tested" in before.lower()) and ("gradual" in before.lower() or "gradually" in before.lower() or "rollout" in before.lower()):
        risk_level = "low"
    elif "rollout" in before.lower():
        risk_level = "medium"

    # Compose AFTER summary (single paragraph, <= limit)
    parts = []
    parts.append("We rolled out a performance improvement aimed at making the product faster and more reliable for customers")
    if metric:
        parts[-1] += f" (about {metric} lower latency)"
    parts[-1] += "."

    # Revenue implication (as an implication, not a guaranteed fact)
    parts.append("This can support revenue goals by improving user experience, which may help retention and conversion.")

    if timeline:
        parts.append(f"The deployment is phased {timeline} with monitoring in place.")

    if risk_level:
        parts.append(f"Risk is considered {risk_level} because testing was completed and the rollout is gradual.")

    # Join into single paragraph and enforce formatting + length constraints
    after = " ".join(parts)
    after = ensure_paragraph_format(after, format_req)
    after = ensure_length(after, length_limit)

    # Differences: compute clear deltas from critique
    differences = [
        "Reframed from engineering implementation details to executive-level business impact and customer experience.",
        "Added revenue implications (retention/conversion) as a cautious implication rather than a guaranteed outcome.",
        "Made timeline explicit using only what the original draft stated.",
        "Added an explicit risk level justified by testing + phased rollout language.",
        "Removed/avoided technical jargon while keeping one concrete metric (if present) for specificity."
    ]

    return {
        "after_summary": after,
        "after_word_count": word_count(after),
        "differences_explained": differences
    }


# -----------------------------
# 5) RUN: produce the full required outputs (visible)
# -----------------------------
print("=" * 80)
print("EXERCISE 3 — SELF-REFLECTION (CRITIQUE-AND-REVISE) OUTPUT")
print("=" * 80)

print("\n1) REQUIREMENTS USED FOR CRITIQUE")
print("-" * 80)
print("Task Objective:", TASK_OBJECTIVE)
print("Target Audience:", TARGET_AUDIENCE)
print("Length Limit:", f"<= {LENGTH_LIMIT_WORDS} words")
print("Format Requirement:", FORMAT_REQUIREMENT)
print("Requirements Checklist:")
for i, r in enumerate(REQUIREMENTS, 1):
    print(f"  {i}. {r}")

print("\n2) BEFORE Summary (verbatim)")
print("-" * 80)
print(BEFORE_SUMMARY)
print(f"\n(BEFORE word count: {word_count(BEFORE_SUMMARY)})")

print("\n3) Self-Reflection Prompt Used (for documentation)")
print("-" * 80)
print(SELF_REFLECTION_PROMPT)

crit = critique_summary(
    before=BEFORE_SUMMARY,
    requirements=REQUIREMENTS,
    audience=TARGET_AUDIENCE,
    length_limit=LENGTH_LIMIT_WORDS,
    format_req=FORMAT_REQUIREMENT,
    task_objective=TASK_OBJECTIVE
)

print("\n4) Criteria Assessment (PASS / PARTIAL / FAIL)")
print("-" * 80)
for row in crit["criteria_assessment"]:
    print(f"- {row['criterion']}: {row['grade']} — {row['explanation']}")

print("\n5) Major Gaps (3–5)")
print("-" * 80)
for g in crit["major_gaps"]:
    print(f"- {g}")

print("\n6) Revision Plan")
print("-" * 80)
for p in crit["revision_plan"]:
    print(f"- {p}")

rev = revise_summary(
    before=BEFORE_SUMMARY,
    critique=crit,
    requirements=REQUIREMENTS,
    audience=TARGET_AUDIENCE,
    length_limit=LENGTH_LIMIT_WORDS,
    format_req=FORMAT_REQUIREMENT,
    task_objective=TASK_OBJECTIVE
)

print("\n7) AFTER Summary (improved)")
print("-" * 80)
print(rev["after_summary"])
print(f"\n(AFTER word count: {rev['after_word_count']} — limit {LENGTH_LIMIT_WORDS})")

print("\n8) Differences Explained (Before vs After)")
print("-" * 80)
for d in rev["differences_explained"]:
    print(f"- {d}")

EXERCISE 3 — SELF-REFLECTION (CRITIQUE-AND-REVISE) OUTPUT

1) REQUIREMENTS USED FOR CRITIQUE
--------------------------------------------------------------------------------
Task Objective: Summarize the announcement for executives, focusing on business impact.
Target Audience: Executives (non-technical)
Length Limit: <= 120 words
Format Requirement: Single paragraph (no bullet points).
Requirements Checklist:
  1. Focus on business impact (not implementation details)
  2. Mention timeline
  3. Mention risk level
  4. Mention revenue implications
  5. Avoid technical jargon and internal engineering terms

2) BEFORE Summary (verbatim)
--------------------------------------------------------------------------------
The engineering team released a new backend optimization that improves database indexing and caching performance.
It reduces API latency by 40% and refactors the query engine for better efficiency.
Several internal components were updated, and the system architecture was modif